# Benchmark publication audit, 23 September 2026

## Summary
Disk-backed OPFS VFS has the lowest persistent-backend median in both SQL workloads in all six groups. On Studio Firefox, SQL-suite medians nearly tie with IndexedDB and ranges overlap. Initialization can favor IndexedDB. This supports a workload-specific comparison, not a universal performance claim.

## Method and sources
Six original JSON exports in `apps/website/public/benchmarks/results/2026-09-23/`, grouped by machine and browser. One warm-up excluded; five measured repetitions per job. Milliseconds, lower is faster. This small audit uses exact-value tables; the website contains full observed ranges and phase details. Run all cells from the repository root or this notebook directory with Python 3, standard library only.

### Assumptions and limits
The source base and collector patch match. OS, Safari/Chrome versions, power settings and free space differ. Compare complete systems. Safari AHP is omitted by protocol. Memory-only PGlite does not persist. Clean reopen does not prove equal crash durability. See the dataset README and provenance for conditions.


In [1]:
from pathlib import Path
import hashlib
import json
from statistics import median

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'apps/website/public/benchmarks/results/2026-09-23').exists())
data = root / 'apps/website/public/benchmarks/results/2026-09-23'
reports = {}
for machine in ['m5-pro', 'm1-max']:
    provenance = json.loads((data / machine / 'provenance.json').read_text())
    for collection in provenance['collections']:
        raw = (data / machine / collection['file']).read_bytes()
        assert hashlib.sha256(raw).hexdigest() == collection['sha256']
        report = json.loads(raw)
        assert report['status'] == 'complete' and report['eligibleForReview'] and not report['interrupted']
        reports[machine, collection['browser']] = report
print('Six raw-file checksums and collection eligibility flags verified.')


Six raw-file checksums and collection eligibility flags verified.


In [2]:
print('Machine | Browser | SQL disk ms | SQL IDB ms | SQL AHP ms | Transaction disk ms | IDB/disk transaction ratio')
for (machine, browser), report in sorted(reports.items()):
    def timing(job):
        values = [e['samples'][0]['workloadMs'] for e in report['entries'] if e['jobId'] == job and e['warmup'] is False]
        if not values:
            return None
        assert len(values) == 5
        return median(values)
    sql = timing('pglite-speedtest-opfs-vfs-disk')
    idb = timing('pglite-speedtest-idb')
    ahp = timing('pglite-speedtest-opfs-ahp')
    tx = timing('transactions-opfs-vfs-disk')
    ratio = timing('transactions-idb') / tx
    ahp_text = 'omitted' if ahp is None else f'{ahp:.2f}'
    print(f'{machine} | {browser} | {sql:.2f} | {idb:.2f} | {ahp_text} | {tx:.2f} | {ratio:.2f}x')


Machine | Browser | SQL disk ms | SQL IDB ms | SQL AHP ms | Transaction disk ms | IDB/disk transaction ratio
m1-max | chrome | 3468.13 | 3780.04 | 3744.55 | 33.84 | 2.37x
m1-max | firefox | 5154.52 | 5181.70 | 7477.88 | 37.52 | 2.90x
m1-max | safari | 3329.10 | 4867.76 | omitted | 27.38 | 6.73x
m5-pro | chrome | 2026.98 | 2315.21 | 2101.34 | 21.52 | 2.40x
m5-pro | firefox | 2786.28 | 2915.62 | 4386.72 | 23.94 | 2.88x
m5-pro | safari | 1711.56 | 2923.02 | omitted | 16.78 | 7.73x
